# Task 4: Financial Risk Identification

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats

In [ ]:
df = pd.read_csv("goldman_sachs_cleaned.csv")

In [3]:
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])

In [4]:
df["Year"] = df["TransactionDate"].dt.year
df["Month"] = df["TransactionDate"].dt.month
df["YearMonth"] = df["TransactionDate"].dt.to_period("M")

In [5]:
credit_types = ["Deposit"]

debit_types = ["Withdrawal", "Payment", "Transfer"]

df["FlowType"] = np.where(
    df["TransactionType"].isin(credit_types),
    "Credit",
    "Debit"
)

In [6]:
df.head()

,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,TransactionAmount,AccountBalance,RiskScore,CreditRating,TenureMonths,IsOverdraft,Year,Month,YearMonth,FlowType
0,33,Cust6549,Acc12334,Credit,Withdrawal,Savings Account,Firm C,Central,Manager 1,2023-10-21,87480.05448,74008.43310,0.729101,319,200,False,2023,10,2023-10,Debit
1,177,Cust2942,Acc52650,Credit,Withdrawal,Home Loan,Firm A,East,Manager 3,2023-06-20,20315.74505,22715.83590,0.472424,692,47,False,2023,6,2023-06,Debit
2,178,Cust6776,Acc45101,Current,Deposit,Personal Loan,Firm C,South,Manager 3,2023-01-02,10484.57165,42706.09210,0.648784,543,109,False,2023,1,2023-01,Credit
3,173,Cust2539,Acc88252,Current,Withdrawal,Mutual Fund,Firm A,Central,Manager 2,2023-07-25,45122.27373,114176.56870,0.734832,430,103,False,2023,7,2023-07,Debit
4,67,Cust2626,Acc21878,Savings,Withdrawal,Home Loan,Firm C,Central,Manager 4,2023-07-25,42360.79878,17863.02644,0.289304,468,234,False,2023,7,2023-07,Debit



# Large Withdrawal Criterion

For this analysis, a Large Withdrawal is defined as any withdrawal transaction whose amount is greater than or equal to the
75th percentile of all withdrawal amounts.

This threshold is chosen to objectively identify unusually large withdrawals while adapting to the distribution of the dataset.



In [7]:
# Identifying Overdraft Transactions

overdraft_transactions = df[df["AccountBalance"] < 0]

print("Total Overdraft Transactions:", len(overdraft_transactions))

overdraft_transactions.head()

Total Overdraft Transactions: 16


,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,TransactionAmount,AccountBalance,RiskScore,CreditRating,TenureMonths,IsOverdraft,Year,Month,YearMonth,FlowType
122,184,Cust8772,Acc16241,Loan,Withdrawal,Mutual Fund,Firm B,Central,Manager 2,2023-02-17,88560.46331,-19222.713000,0.355381,653,147,True,2023,2,2023-02,Debit
161,166,Cust8461,Acc28292,Current,Transfer,Mutual Fund,Firm B,Central,Manager 2,2024-04-16,47644.51746,-2250.087452,0.406043,408,24,True,2024,4,2024-04,Debit
174,121,Cust7730,Acc26973,Current,Payment,Home Loan,Firm B,North,Manager 1,2024-02-09,48164.13332,-8762.711283,0.816948,475,52,True,2024,2,2024-02,Debit
199,111,Cust9066,Acc88449,Loan,Payment,Personal Loan,Firm A,East,Manager 3,2023-01-28,44945.55824,-5315.650952,0.388548,619,168,True,2023,1,2023-01,Debit
215,20,Cust2942,Acc70314,Credit,Withdrawal,Credit Card,Firm D,East,Manager 4,2023-01-09,54119.72962,-31597.850590,0.594390,318,215,True,2023,1,2023-01,Debit


In [8]:
# Counting Overdraft Transactions for Each Account

overdraft_count = (
    overdraft_transactions
    .groupby("AccountID")
    .size()
    .reset_index(name="OverdraftCount")
)

overdraft_count.head()

,AccountID,OverdraftCount
0,Acc16241,1
1,Acc19178,1
2,Acc23736,1
3,Acc26973,1
4,Acc28154,1


In [9]:
# Flagging Overdraft Accounts

frequent_overdraft_accounts = overdraft_count[
    overdraft_count["OverdraftCount"] >= 2
]

print("Number of Frequent Overdraft Accounts:",
      len(frequent_overdraft_accounts))

frequent_overdraft_accounts.head()

Number of Frequent Overdraft Accounts: 1


,AccountID,OverdraftCount
5,Acc28292,2


# Overdraft Analysis

Accounts with a negative account balance were identified as overdraft transactions.

Accounts experiencing two or more overdraft occurrences were flagged as Frequent Overdraft 
Accounts, as repeated overdrafts may indicate financial instability or increased credit risk.


In [10]:
# Calculating Balance Volatility

balance_volatility = (
    df.groupby("AccountID")["AccountBalance"]
      .agg(["mean", "std"])
      .reset_index()
)

# Rename columns
balance_volatility.rename(
    columns={
        "mean": "AverageBalance",
        "std": "BalanceStdDev"
    },
    inplace=True
)

# Calculate Coefficient of Variation (CV)
balance_volatility["BalanceCV"] = (
    balance_volatility["BalanceStdDev"] /
    balance_volatility["AverageBalance"].abs()
)

balance_volatility.head()

,AccountID,AverageBalance,BalanceStdDev,BalanceCV
0,Acc10117,70107.007957,25886.972758,0.369249
1,Acc10996,43568.008084,9434.002316,0.216535
2,Acc11062,38137.132610,3208.737888,0.084137
3,Acc11188,69652.151044,35494.660810,0.509599
4,Acc11285,97401.348560,55922.732441,0.574147


In [11]:
# Calculate CV Threshold

cv_threshold = balance_volatility["BalanceCV"].quantile(0.75)

print("CV Threshold:", cv_threshold)

CV Threshold: 0.5728697824509694


In [12]:
# Flag High Volatility Accounts

high_volatility_accounts = balance_volatility[
    balance_volatility["BalanceCV"] >= cv_threshold
]

print("Number of High Volatility Accounts:",
      len(high_volatility_accounts))

high_volatility_accounts.head()

Number of High Volatility Accounts: 45


,AccountID,AverageBalance,BalanceStdDev,BalanceCV
4,Acc11285,97401.348560,55922.732441,0.574147
5,Acc11837,84852.733695,60694.391957,0.715291
8,Acc13357,69179.806513,40614.664902,0.587088
13,Acc16241,73521.710375,45312.204045,0.616311
17,Acc18140,39960.076965,31325.244170,0.783914


# Balance Volatility Analysis

The balance volatility for each account was measured using the Coefficient of Variation (CV).

CV = Standard Deviation / Mean Balance

Accounts with a CV greater than or equal to the 75th percentile were classified as
High Volatility Accounts, indicating larger fluctuations in account balances and potentially 
higher financial risk.

In [13]:
# Calculating IQR

Q1 = df["TransactionAmount"].quantile(0.25)
Q3 = df["TransactionAmount"].quantile(0.75)

IQR = Q3 - Q1

print("Q1 :", Q1)
print("Q3 :", Q3)
print("IQR:", IQR)

Q1 : 31692.0048
Q3 : 71913.39457
IQR: 40221.38977000001


In [14]:
# Calculate IQR Bounds

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Lower Bound: -28640.079855000015
Upper Bound: 132245.47922500002


In [15]:
# Flagging Transaction Amount Anomalies using IQR

df["IsAnomaly_IQR"] = (
    (df["TransactionAmount"] < lower_bound) |
    (df["TransactionAmount"] > upper_bound)
)

print("Number of IQR Anomalies:",
      df["IsAnomaly_IQR"].sum())

Number of IQR Anomalies: 0


In [16]:
# Displaying detected IQR Anomalies

iqr_anomalies = df[df["IsAnomaly_IQR"]]

iqr_anomalies.head()

,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,...,AccountBalance,RiskScore,CreditRating,TenureMonths,IsOverdraft,Year,Month,YearMonth,FlowType,IsAnomaly_IQR


# IQR-Based Anomaly Detection

Transaction amount anomalies were identified using the Interquartile Range (IQR) method.

The lower and upper bounds were calculated as:

- Lower Bound = Q1 − (1.5 × IQR)
- Upper Bound = Q3 + (1.5 × IQR)

Transactions falling outside these bounds were flagged as anomalies.


In [17]:
# Calculating Z-Score

df["ZScore"] = np.abs(
    stats.zscore(df["TransactionAmount"])
)

df.head()

,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,...,RiskScore,CreditRating,TenureMonths,IsOverdraft,Year,Month,YearMonth,FlowType,IsAnomaly_IQR,ZScore
0,33,Cust6549,Acc12334,Credit,Withdrawal,Savings Account,Firm C,Central,Manager 1,2023-10-21,...,0.729101,319,200,False,2023,10,2023-10,Debit,False,1.266887
1,177,Cust2942,Acc52650,Credit,Withdrawal,Home Loan,Firm A,East,Manager 3,2023-06-20,...,0.472424,692,47,False,2023,6,2023-06,Debit,False,1.148219
2,178,Cust6776,Acc45101,Current,Deposit,Personal Loan,Firm C,South,Manager 3,2023-01-02,...,0.648784,543,109,False,2023,1,2023-01,Credit,False,1.501730
3,173,Cust2539,Acc88252,Current,Withdrawal,Mutual Fund,Firm A,Central,Manager 2,2023-07-25,...,0.734832,430,103,False,2023,7,2023-07,Debit,False,0.256222
4,67,Cust2626,Acc21878,Savings,Withdrawal,Home Loan,Firm C,Central,Manager 4,2023-07-25,...,0.289304,468,234,False,2023,7,2023-07,Debit,False,0.355519


In [18]:
# Flagging Z-Score Anomalies
#Criteria: |Z| > 3

df["IsAnomaly_ZScore"] = df["ZScore"] > 3

print("Number of Z-Score Anomalies:",
      df["IsAnomaly_ZScore"].sum())

Number of Z-Score Anomalies: 0


In [19]:
# Display Z-Score Anomalies

zscore_anomalies = df[df["IsAnomaly_ZScore"]]

zscore_anomalies.head()

,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,...,CreditRating,TenureMonths,IsOverdraft,Year,Month,YearMonth,FlowType,IsAnomaly_IQR,ZScore,IsAnomaly_ZScore


# Z-Score Based Anomaly Detection

A Z-Score was calculated for each transaction amount.

Transactions with an absolute Z-Score greater than 3 were classified as anomalous,
indicating values that significantly deviate from the overall distribution.

In [20]:
# Identifying Suspicious Customers

suspicious_customers = df[
    (df["RiskScore"] > 0.70) &
    (df["CreditRating"] < 450) &
    (df["AccountBalance"] < 0)
][[
    "CustomerID",
    "AccountID",
    "RiskScore",
    "CreditRating",
    "AccountBalance"
]].drop_duplicates()

print("Number of Suspicious Customers:",
      len(suspicious_customers))

suspicious_customers.head()

Number of Suspicious Customers: 0


,CustomerID,AccountID,RiskScore,CreditRating,AccountBalance


In [21]:
# Display Complete List

suspicious_customers 

,CustomerID,AccountID,RiskScore,CreditRating,AccountBalance


# Suspicious Customer Identification

Customers were classified as suspicious if they satisfied all of the following conditions:

- Risk Score greater than 0.70
- Credit Rating below 450
- Negative Account Balance (Overdraft)

These conditions indicate customers with elevated financial risk who may require 
additional monitoring or investigation. But no suspicious accoutns were found in the dataset.

# Task 4 Summary

The following financial risk analyses were performed:

- Identified accounts with frequent large withdrawals.
- Identified accounts with frequent overdrafts.
- Calculated balance volatility using the Coefficient of Variation (CV).
- Detected anomalous transaction amounts using both the IQR and Z-Score methods.
- Identified customers exhibiting suspicious financial behavior based on Risk Score, Credit Rating, and Account Balance.